# Chuẩn bị đầu vào Milestone 2 từ backup Milestone 1

Notebook khôi phục source LaTeX/BibTeX, metadata và reference metadata ở local Colab trước, rồi publish một transport ZIP đã xác minh lên Drive. Đây là bước chuẩn bị raw input; parser, hierarchy, cleanup và dedup M2 sẽ được viết/chạy sau.

**M1 backup → kiểm SHA-256/receipts → raw_m1 + inventory → parser M2 về sau.**

Dùng thư mục backup Drive nguyên vẹn hoặc ZIP tải từ Drive. Notebook nhận cả checkpoint/full backup/session report, không nhầm manifest-only với source thật. Chạy **CPU / accelerator None**, một runtime cho một DATASET_NAME; không chạy hai máy ghi cùng đích.

## 1. Đường dẫn của bạn

Trên Colab, Drive được mount ở bước 2. Sửa SOURCE_PATH theo thư mục backup M1 thật; mặc định là đường dẫn crawler M1 đã dùng. Nếu dùng ZIP, trỏ trực tiếp tới file ZIP trên Drive hoặc local, không cần giải nén.

DATASET_NAME phân biệt tài khoản, ví dụ Danh, Danh1, Danh2, Khanh1. ZIP đã kiểm tra trong phiên phát triển có queue Danh2; đó chỉ là dữ liệu một tài khoản. Cả nhóm có thể giữ các dataset riêng dưới cùng preparationM2/raw_m1, hoặc trỏ SOURCE_PATH tới thư mục cha chứa đủ backup đã gom.

Khi mở notebook từ GitHub trên Colab, repo phải có src/preparation/milestone2.py mới. Commit/push code mới trước khi clone, hoặc đưa checkout mới lên runtime. Không cần cài toàn bộ dependencies của crawler cho bước khôi phục này.
Mặc định local chuẩn bị toàn bộ bài pending trong một lượt. Legacy Drive resume đặt `LOCAL_PREPARATION_DIR=PREPARATION_DIR` và `MAX_PAPERS_PER_RUN=100`; giữ nguyên `SOURCE_PATH`, `PREPARATION_DIR` và `DATASET_NAME`.

`RETRY_BLOCKED=False` giúp ID lỗi không chiếm lại lượt và cản ID mới. Bật True để thử lại các ID lỗi (vẫn tính trong giới hạn mỗi lượt). `MAX_PAPERS_PER_RUN=None` kiểm/khôi phục lại toàn corpus, dùng khi cần sửa dữ liệu đã lưu bị hỏng. Nếu thêm/đổi backup gốc, dùng dataset mới hoặc chạy None để xây lại và cập nhật tiến độ; không dùng tiến độ cũ cho một nguồn đã thay đổi.


In [ ]:
from pathlib import Path

SOURCE_PATH = Path('/content/drive/MyDrive/arXivProject/Milestone1')
PREPARATION_DIR = Path('/content/drive/MyDrive/arXivProject/preparationM2')
DATASET_NAME = 'Danh2'  # Đổi đúng tài khoản/dataset của bạn.
PROJECT_DIR = None     # Local: tự tìm repo; có thể đặt Path('/content/arXivProject').
REPOSITORY_URL = 'https://github.com/Hireophant/arXivProject.git'
REPOSITORY_BRANCH = 'master'
CLONE_IF_NEEDED = True
SHOW_PROGRESS_EVERY = 10  # Báo mỗi N bài hoàn tất/được kiểm.
MAX_PAPERS_PER_RUN = None  # Local mặc định chạy hết pending; Drive legacy resume đặt 100.
RETRY_BLOCKED = False      # True: cho phép chọn lại ID đã làm nhưng bị blocked_input.
LOCAL_PREPARATION_DIR = Path('/content/arXivProject-preparation')  # Raw/work local; đặt = PREPARATION_DIR để tiếp tục batch Drive cũ.
PUBLISH_TRANSPORT = True
PUBLISH_PARTIAL_PILOT = False  # True chỉ cho pilot batch, không phải bàn giao full.
DRIVE_TRANSPORT_FOLDER_ID = ''  # Copy đoạn sau /folders/ trong URL Drive đích; trống thì chỉ giữ bundle local.
LOCAL_DISK_BUDGET_BYTES = 0  # Đặt mức tối thiểu nếu nhóm đã đo corpus; 0 chỉ báo dung lượng còn trống.


## 2. Mở source code và mount Drive

Chỉ clone khi chưa có checkout và đang chạy Colab; không tự pull/ghi đè code đang có. Notebook không gọi arXiv hoặc Semantic Scholar. Quyền Drive chỉ được xin qua hộp thoại mount của Colab khi chính bạn chạy cell này.

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python >= 3.11.')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False

if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'src/preparation/milestone2.py').is_file()), None)
if PROJECT_DIR is None and IN_COLAB:
    PROJECT_DIR = Path('/content/arXivProject')
if PROJECT_DIR is None:
    raise RuntimeError('Mở notebook trong checkout mới hoặc đặt PROJECT_DIR tới project root.')
PROJECT_DIR = Path(PROJECT_DIR).resolve()
if not PROJECT_DIR.exists() and IN_COLAB and CLONE_IF_NEEDED:
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH,
                    REPOSITORY_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB and PROJECT_DIR.exists() and (PROJECT_DIR / '.git').is_dir() and CLONE_IF_NEEDED:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', REPOSITORY_BRANCH], check=False)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', REPOSITORY_BRANCH], check=False)

if not (PROJECT_DIR / 'src/preparation/milestone2.py').is_file():
    raise RuntimeError('Checkout chưa có preparation helper mới. Đưa code mới lên runtime rồi chạy lại.')

# Xóa cache module cũ nếu runtime Colab đã import trước đó
for mod in list(sys.modules.keys()):
    if mod == 'preparation' or mod.startswith('preparation.'):
        del sys.modules[mod]

sys.path.insert(0, str(PROJECT_DIR / 'src'))
from preparation import inspect_backups, prepare_corpus, validate_prepared_corpus
from preparation.transport import pack_prepared_corpus

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', DATASET_NAME):
    raise ValueError('DATASET_NAME dùng chữ không dấu/số/_/-, ví dụ Danh2 hoặc Khanh1.')
LOCAL_PREPARATION_DIR = Path(globals().get('LOCAL_PREPARATION_DIR', '/content/arXivProject-preparation'))
PUBLISH_TRANSPORT = bool(globals().get('PUBLISH_TRANSPORT', True))
PUBLISH_PARTIAL_PILOT = bool(globals().get('PUBLISH_PARTIAL_PILOT', False))
DRIVE_TRANSPORT_FOLDER_ID = str(globals().get('DRIVE_TRANSPORT_FOLDER_ID', ''))
LOCAL_DISK_BUDGET_BYTES = int(globals().get('LOCAL_DISK_BUDGET_BYTES', 0))
DRIVE_OUTPUT_ROOT = Path(PREPARATION_DIR) / 'raw_m1' / DATASET_NAME
OUTPUT_ROOT = LOCAL_PREPARATION_DIR / 'raw_m1' / DATASET_NAME if IN_COLAB else DRIVE_OUTPUT_ROOT
if IN_COLAB and OUTPUT_ROOT == DRIVE_OUTPUT_ROOT:
    PUBLISH_TRANSPORT = False  # Legacy batch resume writes raw files to Drive; do not package there.
print('Python:', sys.version.split()[0])
print('Branch:', REPOSITORY_BRANCH)
print('Source:', SOURCE_PATH)
print('Đích raw local:', OUTPUT_ROOT)
if IN_COLAB:
    import shutil
    free_bytes = shutil.disk_usage(LOCAL_PREPARATION_DIR.parent).free
    if LOCAL_DISK_BUDGET_BYTES and free_bytes < LOCAL_DISK_BUDGET_BYTES:
        raise RuntimeError(f'Không đủ local disk budget: cần {LOCAL_DISK_BUDGET_BYTES} bytes, còn {free_bytes}.')
    print(f'Local disk còn {free_bytes} bytes. Đặt LOCAL_DISK_BUDGET_BYTES sau khi đo corpus; lỗi ENOSPC sẽ không chốt checkpoint.')
    print('Drive chỉ nhận receipt sau khi Drive API xác minh archive.')


## 3. Kiểm kê backup trước khi khôi phục

Preview chỉ đếm receipt và đọc tên queue, **chưa xác nhận source đầy đủ**. Không chọn riêng report.zip hay một corpus.zip bên trong backup; chọn cả thư mục Milestone1 hoặc ZIP tải từ Drive chứa thư mục đó. Giữ backup_receipt.json cạnh corpus.zip.

In [ ]:
preview = inspect_backups(SOURCE_PATH)
print('Loại input:', preview['source_kind'])
print('Số bộ backup:', preview['backup_count'])
print('Corpus ZIP thiếu receipt:', len(preview['orphan_corpus_archives']))
print('Queues/tài khoản:', json.dumps(preview['queues'], ensure_ascii=False, indent=2))
if not preview['backup_count']:
    raise ValueError('Không tìm thấy backup_receipt.json; kiểm tra lại SOURCE_PATH.')
print('Preview chưa xác minh hash. Cell tiếp theo sẽ kiểm và khôi phục.')

## 4. Kiểm hash và khôi phục raw input

Mỗi corpus.zip phải khớp hash/size trong backup receipt; source từng version phải khớp M1 source receipt. Source giữ nguyên bytes, đường dẫn file con và citations. Metadata/references được lưu theo cấu trúc JSON dễ dùng lại.

Chạy lại cùng đích sẽ kiểm lại dữ liệu và phục hồi file đã hỏng; không nhân đôi publication/version. Nếu runtime ngắt, preparation.json còn in_progress: chạy lại để hoàn thành, không dùng kết quả đang dở. Nếu hai bản source hợp lệ của cùng version có nội dung khác nhau, version bị chặn để kiểm thay vì chọn ngẫu nhiên.

Không dùng thư mục đích có file cá nhân sẵn. Notebook chỉ tiếp tục đích đã do chính helper này tạo; M1 và M2 phải là hai cây không chồng nhau.
Sau khi lưu, helper đọc lại source/metadata/references và inventory, ghi input_validation.json + snapshot_id. Kiểm tra không đạt sẽ chặn bàn giao. Một snapshot valid có thể còn version blocked; đó là input partial đã kiểm, không phải toàn corpus đã đủ.

Tiến độ nằm trong `batch_checkpoint.json`, chỉ chốt sau khi đọc lại và kiểm tra output thành công. Runtime ngắt giữa lượt: chạy lại sẽ chọn lại lượt chưa chốt, tận dụng source hợp lệ đã ghi; không nhảy mất ID. Mỗi lượt vẫn đọc/kiểm hash các archive và kiểm lại phần output đã chuẩn bị, nên giới hạn 100 bài không có nghĩa thời gian giảm còn đúng 100/tổng số bài.

ID chưa tới lượt vẫn được ghi vào manifest/inventory với `blocked_input`, reason `PENDING_BATCH`, không có đường dẫn source dùng cho parser. Các ID đã thử nhưng thiếu/hỏng source có reason lỗi riêng và được tính là đã xử lý, chưa chắc đã sẵn sàng. Tiếp tục chạy đến khi số bài chờ bằng 0; nếu còn lỗi input thì vẫn là partial.


In [ ]:
import time

started_at = time.monotonic()
progress_count = 0
def show_progress(message):
    global progress_count
    progress_count += 1
    if message.startswith('[scanning]') or message.startswith('[validating_local]') or message.startswith('[packaging]') or message.startswith('Completed '):
        elapsed = time.monotonic() - started_at
        mins, secs = divmod(int(elapsed), 60)
        print(f'[{mins:02d}:{secs:02d}] {message}', flush=True)

summary = prepare_corpus(SOURCE_PATH, OUTPUT_ROOT, progress=show_progress,
                         progress_every=SHOW_PROGRESS_EVERY, max_papers=MAX_PAPERS_PER_RUN, retry_blocked=RETRY_BLOCKED)
elapsed_total = time.monotonic() - started_at
mins, secs = divmod(int(elapsed_total), 60)
print(f'[local_validated] hoàn tất kiểm tra local sau {mins:02d}:{secs:02d}', flush=True)
if not summary.get('input_valid'):
    raise RuntimeError('Dữ liệu sau lưu không đạt kiểm tra. Xem input_validation.json rồi chạy lại bước chuẩn bị.')
batch = summary['batch']
print(f"Tiến độ: {batch['papers_processed']}/{summary['paper_count']} bài; còn chờ {batch['papers_pending']}; ready {summary['versions_ready_tex']}; no_tex {summary['versions_empty_no_tex']}; blocked {summary['versions_blocked']}.")
if batch['papers_pending']:
    print('⚠️ RAW CHƯA HOÀN TẤT: PENDING_BATCH là bài chưa tới lượt, không phải file mất. Cần chạy lại cell khôi phục để tiếp tục.')
elif summary['versions_blocked']:
    print('⚠️ RAW COVERAGE PARTIAL: không còn bài chờ; các version blocked vẫn được khai báo rõ.')
else:
    print('✅ RAW INPUT READY: mọi version phát hiện đã có source hợp lệ hoặc no_tex có bằng chứng.')

publication = {'state': 'not_requested'}


In [ ]:
# Retry this cell after an upload failure; it never calls prepare_corpus.
publication = {'state': 'not_requested'}
if IN_COLAB and PUBLISH_TRANSPORT:
    if not summary.get('input_valid') or (batch['papers_pending'] and not PUBLISH_PARTIAL_PILOT):
        publication = {'state': 'waiting_for_local_completion'}
        print('[publish] Cần local validation hợp lệ và papers_pending=0 trước; pilot cần PUBLISH_PARTIAL_PILOT=True.')
    else:
        archive_name = f"{DATASET_NAME}-{summary['snapshot_id']}-{summary['run_id'][:8]}.zip"
        local_archive = LOCAL_PREPARATION_DIR / 'transport' / archive_name
        if not local_archive.exists():
            print('[Đóng gói] Tạo bundle ZIP local...', flush=True)
            pack_prepared_corpus(OUTPUT_ROOT, local_archive, progress=show_progress)
        
        state = 'remote_verified_pilot' if batch['papers_pending'] else 'remote_verified'
        
        # Tùy chọn 1: Nếu người dùng có cấu hình DRIVE_TRANSPORT_FOLDER_ID -> Upload qua Drive API resumable upload
        if DRIVE_TRANSPORT_FOLDER_ID:
            try:
                from google.colab import auth
                from googleapiclient.discovery import build
                from googleapiclient.http import MediaFileUpload
                from preparation.publisher import publish_prepared_archive
                auth.authenticate_user()
                def publish_progress(stage, sent, total):
                    pct = int(sent * 100 / total) if total else 0
                    mb_sent = sent / (1024 * 1024)
                    mb_tot = total / (1024 * 1024)
                    if stage == 'uploading':
                        print(f'[Upload Drive API] {pct}% ({mb_sent:.1f}MB / {mb_tot:.1f}MB)', flush=True)
                    elif stage == 'verifying_remote':
                        print('[Drive API] Đang xác minh kích thước và MD5 checksum trên server Drive...', flush=True)
                    elif stage == 'remote_verified':
                        print('[Drive API] Đã xác minh bản trên Drive và công bố receipt!', flush=True)
                    else:
                        print(f'[{stage}] {sent}/{total} bytes', flush=True)
                media_factory = lambda path, resumable: MediaFileUpload(path, resumable=resumable, chunksize=8 * 1024 * 1024)
                published = publish_prepared_archive(local_archive, DRIVE_TRANSPORT_FOLDER_ID, build('drive', 'v3'), media_factory, progress=publish_progress)
                publication = {'state': state, 'pilot': bool(batch['papers_pending']), 'processed': batch['papers_processed'], 'pending': batch['papers_pending'], 'method': 'drive_api', **published}
                print(f'[{state}] processed {batch["papers_processed"]}/{summary["paper_count"]}; pending {batch["papers_pending"]}', publication)
            except BaseException:
                publication = {'state': 'persistence_failed', 'local_bundle': str(local_archive)}
                print('[persistence_failed] Giữ bundle local và chạy lại cell publish:', local_archive)
                raise
        # Tùy chọn 2 (Mặc định cho cả nhóm): Lưu thẳng vào thư mục PREPARATION_DIR/transport trên Google Drive đã mount!
        else:
            try:
                import shutil
                import hashlib
                drive_transport_dir = Path(PREPARATION_DIR) / 'transport'
                drive_transport_dir.mkdir(parents=True, exist_ok=True)
                drive_archive = drive_transport_dir / archive_name
                drive_receipt = drive_transport_dir / (archive_name + '.receipt.json')
                
                print(f'[Drive Mount] Đang lưu 1 file ZIP sang Google Drive: {drive_archive}...', flush=True)
                shutil.copyfile(local_archive, drive_archive)
                
                print('[Drive Mount] Đang đọc lại từ Google Drive để xác minh checksum SHA-256...', flush=True)
                with drive_archive.open('rb') as stream:
                    remote_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
                
                local_receipt = json.loads(Path(str(local_archive) + '.receipt.json').read_text(encoding='utf-8'))
                if remote_sha256 != local_receipt['sha256']:
                    raise ValueError(f'Lỗi đồng bộ Drive mount: SHA-256 không khớp! Local: {local_receipt["sha256"]}, Drive: {remote_sha256}')
                
                # Công bố receipt đã xác minh trực tiếp lên Drive
                verified_receipt = dict(local_receipt, remote_verified=True, drive_path=str(drive_archive),
                                        verified_method='drive_mount_checksum')
                drive_receipt.write_text(json.dumps(verified_receipt, ensure_ascii=False, indent=2), encoding='utf-8')
                
                publication = {
                    'state': state,
                    'pilot': bool(batch['papers_pending']),
                    'processed': batch['papers_processed'],
                    'pending': batch['papers_pending'],
                    'method': 'drive_mount',
                    'archive_path': str(drive_archive),
                    'archive': {'name': archive_name, 'path': str(drive_archive), 'id': ''},
                    'sha256': local_receipt['sha256'],
                    'snapshot_id': local_receipt['snapshot_id']
                }
                print('✅ [Drive Mount] Đã xác minh bản trên Drive thành công!')
                print(f'[{state}] processed {batch["papers_processed"]}/{summary["paper_count"]}; pending {batch["papers_pending"]}')
            except BaseException as e:
                publication = {'state': 'persistence_failed', 'local_bundle': str(local_archive), 'error': str(e)}
                print(f'❌ [persistence_failed] Lỗi khi lưu lên Drive: {e}', flush=True)
                raise


## 5. Xem bài/version đã chuẩn bị

ready_tex: source có TeX đã kiểm; empty_no_tex: M1 xác nhận không có TeX; blocked_input: thiếu/hỏng hoặc source xung đột. Đây là trạng thái **input**, không phải parser M2 đã chạy thành công. Lỗi Semantic Scholar không tự loại bài có source TeX hợp lệ.
`PENDING_BATCH` là bài chưa tới lượt, khác lỗi source thật. Bảng đếm toàn bộ phạm vi backup, không chỉ lượt hiện tại; `batch.selected_ids` ghi chính xác các ID của lượt này. Snapshot thay đổi khi có thêm bài sẵn sàng, cần lấy lại snapshot mới trước mỗi lượt parse.


In [ ]:
import csv
from html import escape

with (OUTPUT_ROOT / 'inventory.csv').open(encoding='utf-8', newline='') as stream:
    inventory = list(csv.DictReader(stream))
pending = [r for r in inventory if r['reason'] == 'PENDING_BATCH']
blocked = [r for r in inventory if r['status'] == 'blocked_input' and r['reason'] != 'PENDING_BATCH']
def show_rows(rows, limit=20):
    shown = rows[:limit]
    fields = ['arxiv_id', 'version', 'status', 'reason', 'm1_error']
    try:
        from IPython.display import HTML, display
    except ImportError:
        for row in shown:
            print({k: row[k] for k in fields})
        return
    html = '<table><thead><tr>' + ''.join('<th>' + escape(k) + '</th>' for k in fields) + '</tr></thead><tbody>'
    html += ''.join('<tr>' + ''.join('<td>' + escape(str(row[k])) + '</td>' for k in fields) + '</tr>' for row in shown)
    display(HTML(html + '</tbody></table>'))
print(f"{summary['paper_count']} bài; {len(inventory)} versions; {len(blocked)} versions lỗi; {len(pending)} versions chưa tới lượt.")
show_rows(inventory)
if blocked:
    print('Các version bị chặn (tối đa 20 dòng):')
    show_rows(blocked)

## 6. Bàn giao cho phần code parser M2

Cấu trúc đã lưu:

```text
preparationM2/raw_m1/<DATASET_NAME>/
  manifest.json                  # schema m2-input-1.0, papers + expected versions + statuses
  preparation.json               # in_progress / ready / partial / failed, run_id
  batch_checkpoint.json          # tiến độ ID đã chốt sau validation; giữ nguyên để tiếp tục
  input_validation.json          # kiểm đọc lại, run_id và snapshot_id
  inventory.csv                  # đúng một dòng cho mỗi publication/version
  blocked_versions.csv
  backup_audit.jsonl              # hash archive và các lỗi input
  provenance.jsonl                # backup/fingerprint của từng source copy
  <paper_key>/
    metadata.json
    reference_result.json        # kết quả references M1 đầy đủ đã chọn
    references.json
    references_all.json
    sources/.vN.receipt.json
    sources/vN/**/*.tex, **/*.bib
```

Parser sau này đọc manifest và các source_dir có trạng thái phù hợp; no_tex không sinh fake elements, blocked vẫn giữ trong coverage. Schema chuẩn bị này khác schema checkpoint crawler 3.0: không đưa lại vào crawler như một run M1 và không sửa source ở đây. Các bảng elements/hierarchy/bibtex M2 nên lưu dưới `Milestone2/parsed/<DATASET_NAME>/` (thư mục kết quả riêng, ngoài preparationM2).

Một người có thể code parser trên mẫu raw đã kiểm ngay; muốn xác nhận bao phủ toàn nhóm phải chuẩn bị đủ dataset của bốn người/12 accounts rồi đối chiếu inventory chung. Không cộng checkpoint/retry thành bài mới. Nếu thiếu backup, lấy lại đúng ZIP/receipt gốc; notebook không tự crawl bù.
Giữ preparationM2 chỉ để đọc khi làm parser. Chạy lại validate_prepared_corpus(RAW_CORPUS_ROOT) trước mỗi lượt parse; không chỉ tin một file marker cũ sau khi source đã đổi. Nhóm ghi snapshot_id vào config/log/test evidence để cùng đối chiếu một bộ raw input.

Khi còn `PENDING_BATCH`, raw snapshot chỉ sẵn sàng để phát triển/thử parser trên phần đã có; chưa được báo là toàn bộ corpus đã đủ. Parser phải giữ các version chờ ở coverage dưới `blocked_input`, reason `PENDING_BATCH`, không đọc metadata/source chưa được khôi phục và không bỏ ID khỏi mẫu số metrics. Record/metadata trong manifest của ID chờ chỉ mô tả phạm vi backup, không xác nhận file metadata đã được ghi. Bước đóng gói/chuyển raw cũng giữ nguyên manifest và inventory này.


In [ ]:
prepared_manifest = json.loads((OUTPUT_ROOT / 'manifest.json').read_text(encoding='utf-8'))
completion = json.loads((OUTPUT_ROOT / 'preparation.json').read_text(encoding='utf-8'))
if completion['run_id'] != prepared_manifest['run_id'] or completion['status'] == 'in_progress':
    raise RuntimeError('Output chưa hoàn tất; chạy lại cell khôi phục trước khi bàn giao.')
input_check = validate_prepared_corpus(OUTPUT_ROOT)
saved_check = json.loads((OUTPUT_ROOT / 'input_validation.json').read_text(encoding='utf-8'))
if (not input_check['valid'] or not saved_check.get('valid') or saved_check['run_id'] != completion['run_id']
        or input_check['snapshot_id'] != saved_check.get('snapshot_id')
        or input_check['snapshot_id'] != completion.get('snapshot_id')):
    raise RuntimeError('Input hoặc validation receipt đã đổi; chưa được bàn giao. Chạy lại cell khôi phục.')
assert len(inventory) == summary['versions_ready_tex'] + summary['versions_empty_no_tex'] + summary['versions_blocked']
RAW_CORPUS_ROOT = OUTPUT_ROOT

is_remote_verified = publication.get('state') in {'remote_verified', 'remote_verified_pilot'}
papers_pending = summary['batch']['papers_pending']
blocked_count = summary['versions_blocked']

print('=' * 72)
print('BÁO CÁO TRẠNG THÁI BÀN GIAO MILESTONE 2 (Tách riêng 4 chỉ số cốt lõi):')
print(f"  1. valid:            {input_check['valid']} (dữ liệu đã ghi khớp snapshot toàn vẹn)")
print(f"  2. papers_pending:   {papers_pending} (số bài còn chờ lượt sau)")
print(f"  3. blocked:          {blocked_count} (số version bị chặn do lỗi nguồn M1)")
print(f"  4. remote_verified:  {is_remote_verified} (bản trên Drive đã xác minh checksum)")
print('=' * 72)

if papers_pending > 0:
    print('⛔ [BÀN GIAO CHƯA HOÀN TẤT - CÒN BÀI CHỜ]')
    print(f'   Hiện còn {papers_pending} bài chưa tới lượt.')
    print('   Tuy valid=True nhưng dữ liệu chưa đủ toàn bộ corpus. Không dùng bản này làm bộ hoàn chỉnh.')
    print('   -> Hãy chạy lại cell khôi phục để xử lý nốt các bài còn lại.')
elif blocked_count > 0:
    print('⚠️ [ĐÃ XỬ LÝ HẾT BÀI - CÓ VERSION LỖI NGUỒN ĐÃ KHAI BÁO]')
    print(f'   Đã xử lý 100% bài (papers_pending=0). Có {blocked_count} version lỗi nguồn M1.')
    print('   Các version lỗi được ghi rõ trong blocked_versions.csv, không bị giấu. Đủ điều kiện cho M2.')
else:
    print('✅ [BÀN GIAO HOÀN HẢO - READY FOR M2]')
    print('   Mọi version phát hiện đều có source hợp lệ hoặc verified no_tex, không có lỗi.')

if is_remote_verified:
    print('\n📋 CẤU HÌNH CHO NOTEBOOK MILESTONE 2 (milestone2.ipynb):')
    if publication.get('method') == 'drive_api' and publication.get('archive', {}).get('id'):
        print(f"PREPARED_REMOTE_FILE_ID = '{publication['archive']['id']}'")
        print(f"PREPARED_SHA256 = '{publication['sha256']}'")
        print(f"PREPARED_SNAPSHOT_ID = '{publication['snapshot_id']}'")
    else:
        archive_path_val = publication.get('archive_path') or str(PREPARATION_DIR / 'transport' / f"{DATASET_NAME}-{summary['snapshot_id']}-{summary['run_id'][:8]}.zip")
        print(f"PREPARED_ARCHIVE = Path('{archive_path_val}')")
    print(f"PREPARED_PAPERS_PENDING = {publication['pending']}")
    print("STAGE_INPUT = True")
    if publication.get('pilot'):
        print("ALLOW_PARTIAL_PILOT = True  # Cần bật cờ này vì đây là pilot batch")
elif IN_COLAB:
    print('\n⚠️ Chưa có bản transport remote_verified trên Drive. Vui lòng kiểm tra cell publish.')
